# 📊 04 - SQL Analytics | Business Insights

---

## 🇧🇷 Português

### 📌 Objetivo

Este notebook representa a etapa de **SQL Analytics** do projeto Amazon ETL Pipeline.

Após a construção das camadas **Bronze, Silver e Gold**, os dados estão preparados para serem explorados por meio de consultas SQL, com foco na geração de **KPIs, análises exploratórias e insights de negócio**.

Nesta etapa, o objetivo não é transformar novamente os dados, mas utilizar as tabelas da camada Gold para responder perguntas relevantes de negócio.

---

### 🏗 Arquitetura

```text
Bronze
   │
   ▼
Silver
   │
   ▼
Gold
   │
   ▼
SQL Analytics
   │
   ├── KPIs
   ├── Category Analysis
   ├── Price Analysis
   ├── Discount Analysis
   ├── Rating Analysis
   ├── Best Seller Analysis
   └── Business Insights
   │
   ▼
Dashboard
```

---

### 🛠 Tecnologias

- Databricks
- Databricks SQL
- SQL
- Delta Lake

---

### 📂 Fontes

As análises serão realizadas utilizando as tabelas da camada Gold:

- `gold.product_analysis`
- `gold.category_summary`
- `gold.best_sellers`
- `gold.discount_analysis`

---

### 🎯 Objetivos Analíticos

Este notebook busca responder perguntas como:

- Quantos produtos existem na base?
- Quantas categorias estão disponíveis?
- Qual é o preço médio dos produtos?
- Qual é o desconto médio?
- Quais categorias possuem mais produtos?
- Quais categorias possuem melhor avaliação?
- Quais categorias apresentam maiores descontos?
- Produtos Best Seller possuem melhores avaliações?
- Quais são os produtos com maior Weighted Rating?
- Como os produtos estão distribuídos por faixa de preço?
- Existe relação entre avaliações e número de reviews?

---

## 🇺🇸 English

### 📌 Purpose

This notebook represents the **SQL Analytics** stage of the Amazon ETL Pipeline.

After implementing the **Bronze, Silver and Gold layers**, the curated datasets are explored using SQL to generate **KPIs, analytical metrics and business insights**.

The main goal is to answer business questions using the Gold Layer without modifying the underlying datasets.

---

### 🎯 Expected Result

Generate analytical insights that will be used as the foundation for the project's **Business Intelligence Dashboard**.

In [ ]:
%sql

SHOW TABLES IN gold;

In [ ]:
%sql

DESCRIBE gold.product_analysis;

In [ ]:
%sql

SELECT *
FROM gold.product_analysis
LIMIT 10;

# 📊 1. General KPIs | Indicadores Gerais

## 🇧🇷 Português

Análise dos principais indicadores gerais da base de produtos, incluindo quantidade de produtos, categorias, preços, descontos, avaliações e volume de compras.

---

## 🇺🇸 English

Analysis of the main general indicators of the product dataset, including products, categories, prices, discounts, ratings and purchase volume.

In [ ]:
%sql

SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT category_name) AS total_categories,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(discount_percentage), 2) AS average_discount,
    ROUND(AVG(stars), 2) AS average_rating,
    SUM(boughtInLastMonth) AS total_bought_last_month
FROM gold.product_analysis;

# 📦 2. Category Analysis | Análise por Categoria

## 🇧🇷 Português

Análise da distribuição e concentração dos produtos entre as diferentes categorias.

---

## 🇺🇸 English

Analysis of product distribution and concentration across different categories.

In [ ]:
%sql

SELECT
    category_name,
    COUNT(*) AS total_products
FROM gold.product_analysis
GROUP BY category_name
ORDER BY total_products DESC
LIMIT 20;

In [ ]:
%sql

WITH category_products AS (

    SELECT
        category_name,
        COUNT(*) AS total_products

    FROM gold.product_analysis

    GROUP BY category_name
)

SELECT
    category_name,
    total_products,
    ROUND(
        total_products * 100.0 /
        SUM(total_products) OVER (),
        2
    ) AS percentage_of_catalog

FROM category_products

ORDER BY total_products DESC
LIMIT 20;

# 💰 3. Price Analysis | Análise de Preços

## 🇧🇷 Português

Análise da distribuição de preços e comparação dos valores entre diferentes categorias e faixas de preço.

---

## 🇺🇸 English

Analysis of price distribution and comparison across different categories and price ranges.

In [ ]:
%sql

SELECT
    price_range,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS average_price
FROM gold.product_analysis
GROUP BY price_range
ORDER BY total_products DESC;

In [ ]:
%sql

SELECT
    category_name,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS average_price
FROM gold.product_analysis
GROUP BY category_name
HAVING COUNT(*) >= 100
ORDER BY average_price DESC
LIMIT 20;

# 💸 4. Discount Analysis | Análise de Descontos

## 🇧🇷 Português

Análise dos descontos aplicados aos produtos e sua distribuição entre categorias e faixas de preço.

---

## 🇺🇸 English

Analysis of product discounts and their distribution across categories and price ranges.

In [ ]:
%sql

SELECT
    price_range,
    COUNT(*) AS total_products,
    ROUND(AVG(discount_percentage), 2) AS average_discount,
    ROUND(MAX(discount_percentage), 2) AS maximum_discount
FROM gold.product_analysis
GROUP BY price_range
ORDER BY average_discount DESC;

In [ ]:
%sql

SELECT
    category_name,
    COUNT(*) AS total_products,
    ROUND(AVG(discount_percentage), 2) AS average_discount
FROM gold.product_analysis
GROUP BY category_name
HAVING COUNT(*) >= 100
ORDER BY average_discount DESC
LIMIT 20;

In [ ]:
%sql

SELECT
    asin,
    title,
    category_name,
    price,
    listPrice,
    discount_amount,
    discount_percentage
FROM gold.product_analysis
WHERE discount_percentage > 0
ORDER BY discount_percentage DESC
LIMIT 20;

# ⭐ 5. Rating Analysis | Análise de Avaliações

## 🇧🇷 Português

Análise das avaliações dos produtos, considerando ratings, categorias e Weighted Rating.

---

## 🇺🇸 English

Analysis of product ratings considering ratings, categories and Weighted Rating.

In [ ]:
%sql

SELECT
    rating_category,
    COUNT(*) AS total_products,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM gold.product_analysis
GROUP BY rating_category
ORDER BY total_products DESC;

In [ ]:
%sql

SELECT
    category_name,
    COUNT(*) AS total_products,
    ROUND(AVG(stars), 2) AS average_rating
FROM gold.product_analysis
GROUP BY category_name
HAVING COUNT(*) >= 100
ORDER BY average_rating DESC
LIMIT 20;

In [ ]:
%sql

SELECT
    asin,
    title,
    category_name,
    stars,
    reviews,
    weighted_rating
FROM gold.product_analysis
ORDER BY weighted_rating DESC
LIMIT 20;

# 🏆 6. Best Seller Analysis | Análise de Best Sellers

## 🇧🇷 Português

Análise dos produtos classificados como Best Sellers e comparação com os demais produtos do catálogo.

---

## 🇺🇸 English

Analysis of products classified as Best Sellers and comparison with the remaining products in the catalog.

In [ ]:
%sql

SELECT
    best_seller_flag,
    COUNT(*) AS total_products,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM gold.product_analysis
GROUP BY best_seller_flag
ORDER BY total_products DESC;

In [ ]:
%sql

SELECT
    best_seller_flag,
    COUNT(*) AS total_products,
    ROUND(AVG(stars), 2) AS average_rating,
    ROUND(AVG(reviews), 2) AS average_reviews,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(discount_percentage), 2) AS average_discount
FROM gold.product_analysis
GROUP BY best_seller_flag
ORDER BY average_rating DESC;

# 🔥 7. Popularity Analysis | Análise de Popularidade

## 🇧🇷 Português

Análise da popularidade dos produtos considerando número de reviews e volume de compras.

---

## 🇺🇸 English

Analysis of product popularity based on review count and purchase volume.

In [ ]:
%sql

SELECT
    asin,
    title,
    category_name,
    stars,
    reviews,
    boughtInLastMonth
FROM gold.product_analysis
ORDER BY reviews DESC
LIMIT 20;

In [ ]:
%sql

SELECT
    asin,
    title,
    category_name,
    price,
    stars,
    boughtInLastMonth
FROM gold.product_analysis
ORDER BY boughtInLastMonth DESC
LIMIT 20;

# 🔎 8. Price x Popularity Analysis | Preço x Popularidade

## 🇧🇷 Português

Análise da relação entre as faixas de preço dos produtos e seu volume de compras.

---

## 🇺🇸 English

Analysis of the relationship between product price ranges and purchase volume.

In [ ]:
%sql

SELECT
    price_range,
    COUNT(*) AS total_products,
    SUM(boughtInLastMonth) AS total_bought,
    ROUND(AVG(boughtInLastMonth), 2) AS average_bought_per_product
FROM gold.product_analysis
GROUP BY price_range
ORDER BY total_bought DESC;

# 🎯 9. Category Performance | Desempenho por Categoria

## 🇧🇷 Português

Análise consolidada do desempenho das categorias considerando produtos, preços, descontos, avaliações e volume de compras.

---

## 🇺🇸 English

Consolidated analysis of category performance based on products, prices, discounts, ratings and purchase volume.

In [ ]:
%sql

SELECT
    category_name,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(discount_percentage), 2) AS average_discount,
    ROUND(AVG(stars), 2) AS average_rating,
    SUM(boughtInLastMonth) AS total_bought
FROM gold.product_analysis
GROUP BY category_name
HAVING COUNT(*) >= 100
ORDER BY total_bought DESC
LIMIT 20;

# 🥇 10. Category Ranking | Ranking de Categorias

## 🇧🇷 Português

Criação de rankings de categorias utilizando métricas de desempenho e Window Functions.

---

## 🇺🇸 English

Creation of category rankings using performance metrics and Window Functions.

In [ ]:
%sql

WITH category_metrics AS (

    SELECT
        category_name,
        COUNT(*) AS total_products,
        ROUND(AVG(stars), 2) AS average_rating,
        ROUND(AVG(discount_percentage), 2) AS average_discount,
        SUM(boughtInLastMonth) AS total_bought

    FROM gold.product_analysis

    GROUP BY category_name

    HAVING COUNT(*) >= 100
)

SELECT
    category_name,
    total_products,
    average_rating,
    average_discount,
    total_bought,

    RANK() OVER (
        ORDER BY total_bought DESC
    ) AS purchase_rank

FROM category_metrics

ORDER BY purchase_rank
LIMIT 20;

# 🧠 11. Business Insights | Insights de Negócio

## 🇧🇷 Português

Análise dos resultados para identificar padrões, tendências e oportunidades de negócio.

---

## 🇺🇸 English

Analysis of the results to identify business patterns, trends and opportunities.

In [ ]:
%sql

SELECT
    category_name,

    COUNT(*) AS total_products,

    ROUND(AVG(price), 2) AS average_price,

    ROUND(AVG(discount_percentage), 2) AS average_discount,

    ROUND(AVG(stars), 2) AS average_rating,

    ROUND(AVG(reviews), 0) AS average_reviews,

    SUM(boughtInLastMonth) AS total_bought

FROM gold.product_analysis

GROUP BY category_name

HAVING COUNT(*) >= 100

ORDER BY total_bought DESC

LIMIT 20;

# 📊 12. Dashboard Data Preparation | Preparação para Dashboard

## 🇧🇷 Português

Preparação das views analíticas que serão utilizadas como fonte de dados para o Dashboard.

---

## 🇺🇸 English

Preparation of analytical views that will be used as data sources for the Dashboard.

In [ ]:
%sql

CREATE OR REPLACE VIEW gold.vw_dashboard_kpis AS

SELECT
    COUNT(*) AS total_products,
    COUNT(DISTINCT category_name) AS total_categories,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(discount_percentage), 2) AS average_discount,
    ROUND(AVG(stars), 2) AS average_rating,
    SUM(boughtInLastMonth) AS total_bought
FROM gold.product_analysis;

In [ ]:
%sql

CREATE OR REPLACE VIEW gold.vw_dashboard_categories AS

SELECT
    category_name,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(discount_percentage), 2) AS average_discount,
    ROUND(AVG(stars), 2) AS average_rating,
    SUM(boughtInLastMonth) AS total_bought
FROM gold.product_analysis
GROUP BY category_name
HAVING COUNT(*) >= 100;

In [ ]:
%sql

CREATE OR REPLACE VIEW gold.vw_dashboard_price_range AS

SELECT
    price_range,
    COUNT(*) AS total_products,
    ROUND(AVG(price), 2) AS average_price,
    ROUND(AVG(discount_percentage), 2) AS average_discount,
    SUM(boughtInLastMonth) AS total_bought
FROM gold.product_analysis
GROUP BY price_range;

In [ ]:
%sql

CREATE OR REPLACE VIEW gold.vw_dashboard_rating AS

SELECT
    rating_category,
    COUNT(*) AS total_products,
    ROUND(AVG(stars), 2) AS average_rating
FROM gold.product_analysis
GROUP BY rating_category;

# 🔍 13. View Validation | Validação das Views

## 🇧🇷 Português

Validação das views analíticas criadas para garantir a consistência dos dados antes da construção do Dashboard.

---

## 🇺🇸 English

Validation of the analytical views to ensure data consistency before building the Dashboard.

In [ ]:
%sql

SELECT *
FROM gold.vw_dashboard_kpis;

In [ ]:
%sql

SELECT *
FROM gold.vw_dashboard_categories
ORDER BY total_bought DESC
LIMIT 10;

In [ ]:
%sql

SELECT *
FROM gold.vw_dashboard_price_range
ORDER BY total_products DESC;

In [ ]:
%sql

SELECT *
FROM gold.vw_dashboard_price_range
ORDER BY total_products DESC;

# ✅ Conclusion | Conclusão

## 🇧🇷 Português

A etapa de **SQL Analytics** foi concluída utilizando as tabelas da camada Gold.

Foram desenvolvidas análises relacionadas a:

- Volume de produtos
- Categorias
- Preços
- Descontos
- Avaliações
- Best Sellers
- Popularidade
- Volume de compras
- Rankings
- Indicadores de negócio

Também foram preparadas views analíticas que serão utilizadas como fonte para a próxima etapa do projeto.

### 🚀 Próxima etapa

**05 - Dashboard**

Os resultados obtidos nesta etapa serão transformados em visualizações interativas e KPIs para facilitar a interpretação dos dados e apoiar decisões de negócio.

---

## 🇺🇸 English

The **SQL Analytics** stage was completed using the Gold Layer datasets.

The analysis covered:

- Product volume
- Categories
- Pricing
- Discounts
- Ratings
- Best Sellers
- Popularity
- Purchase volume
- Rankings
- Business KPIs

Analytical views were also created to support the next stage of the project.

### 🚀 Next Step

**05 - Dashboard**

The analytical results will be transformed into interactive visualizations and KPIs for business intelligence and decision-making.